In [15]:
import torch
from SuperGlue import SuperGlue
import numpy as np
import cv2 as cv
import onnxruntime as ort

input_data_path = "../../res/npz/input_data.npz"
before_preprocess_path = "../../output/before_preprocess.npz"
end_preprocess_path = "../../output/end_preprocess.npz"
end_inference_path = "../../output/end_inference.npz"
model_path = "../../res/weights/superglue_indoor.onnx"


In [16]:
# 加载onnx session, 加载不同的数据
session = ort.InferenceSession(model_path)
input_data = np.load(input_data_path)
before_preprocess = np.load(before_preprocess_path)
end_preprocess = np.load(end_preprocess_path)
end_inference = np.load(end_inference_path)

In [17]:
# 使用 SuperGlue 进行预处理操作
kpt0, kpt1 = input_data["keypoints0"], input_data["keypoints1"]
desc0, desc1 = input_data["descriptors0"], input_data["descriptors1"]
score0, score1 = input_data["scores0"], input_data["scores1"]
image0, image1 = input_data["image0"], input_data["image1"]

kpt0, kpt1 = SuperGlue.preprocess(torch.from_numpy(kpt0), torch.from_numpy(kpt1), image0.shape, image1.shape)
kpt0 = kpt0.numpy()
kpt1 = kpt1.numpy()

In [18]:
# 验证 end_preprocess_path 文件中的内容是否对应成功

# 比较预处理后的关键点
print("验证 kpts0:")
print("kpt0 shape:", kpt0.shape)
print("end_preprocess['kpts0'] shape:", end_preprocess['kpts0'].shape)
print("kpts0 匹配:", np.allclose(kpt0, end_preprocess['kpts0']))

print("\n验证 kpts1:")
print("kpt1 shape:", kpt1.shape)
print("end_preprocess['kpts1'] shape:", end_preprocess['kpts1'].shape)
print("kpts1 匹配:", np.allclose(kpt1, end_preprocess['kpts1']))

# 比较描述符（需要转置，因为C++中desc0是转置后的）
print("\n验证 desc0:")
print("desc0 shape:", desc0.shape)
print("end_preprocess['desc0'] shape:", end_preprocess['desc0'].shape)
print("desc0 匹配:", np.allclose(desc0, end_preprocess['desc0']))

print("\n验证 desc1:")
print("desc1 shape:", desc1.shape)
print("end_preprocess['desc1'] shape:", end_preprocess['desc1'].shape)
print("desc1 匹配:", np.allclose(desc1, end_preprocess['desc1']))

# 比较分数
print("\n验证 scores0:")
print("score0 shape:", score0.shape)
print("end_preprocess['scores0'] shape:", end_preprocess['scores0'].shape)
print("scores0 匹配:", np.allclose(score0, end_preprocess['scores0']))

print("\n验证 scores1:")
print("score1 shape:", score1.shape)
print("end_preprocess['scores1'] shape:", end_preprocess['scores1'].shape)
print("scores1 匹配:", np.allclose(score1, end_preprocess['scores1']))

验证 kpts0:
kpt0 shape: (1, 239, 2)
end_preprocess['kpts0'] shape: (239, 2)
kpts0 匹配: True

验证 kpts1:
kpt1 shape: (1, 244, 2)
end_preprocess['kpts1'] shape: (244, 2)
kpts1 匹配: True

验证 desc0:
desc0 shape: (1, 256, 239)
end_preprocess['desc0'] shape: (256, 239)
desc0 匹配: True

验证 desc1:
desc1 shape: (1, 256, 244)
end_preprocess['desc1'] shape: (256, 244)
desc1 匹配: True

验证 scores0:
score0 shape: (1, 239)
end_preprocess['scores0'] shape: (239,)
scores0 匹配: True

验证 scores1:
score1 shape: (1, 244)
end_preprocess['scores1'] shape: (244,)
scores1 匹配: True


In [19]:
# 验证 inference 后的结果是否正确
inputs = {
    "kpts0": end_preprocess["kpts0"].reshape(1, -1, 2).astype(np.float32),
    "kpts1": end_preprocess["kpts1"].reshape(1, -1, 2).astype(np.float32),
    "desc0": end_preprocess["desc0"].reshape(1, 256, -1).astype(np.float32),
    "desc1": end_preprocess["desc1"].reshape(1, 256, -1).astype(np.float32),
    "scores0": end_preprocess["scores0"].reshape(1, -1).astype(np.float32),
    "scores1": end_preprocess["scores1"].reshape(1, -1).astype(np.float32),
}

outputs = session.run(None, inputs)

indices0_pred = outputs[0]
mscores0_pred = outputs[1]

print("验证 indices0:")
print("indices0_pred shape:", indices0_pred.shape)
print("end_inference['indices0'] shape:", end_inference['indices0'].shape)
print("indices0 匹配:", np.allclose(indices0_pred, end_inference['indices0']))

print("\n验证 mscores0:")
print("mscores0_pred shape:", mscores0_pred.shape)
print("end_inference['mscores0'] shape:", end_inference['mscores0'].shape)
print("mscores0 匹配:", np.allclose(mscores0_pred, end_inference['mscores0']))

验证 indices0:
indices0_pred shape: (1, 239)
end_inference['indices0'] shape: (239,)
indices0 匹配: True

验证 mscores0:
mscores0_pred shape: (1, 239)
end_inference['mscores0'] shape: (239,)
mscores0 匹配: True


In [21]:
indices0_pred

array([[  3,   4,   5,   7,   8,   9,  11,  10,  -1,  14,  13,  15,  16,
         19,  17,  18,  20,  -1,  22,  21,  25,  23,  24,  -1,  -1,  26,
         27,  -1,  29,  30,  31,  33,  35,  32,  37,  38,  36,  39,  40,
         43,  -1,  -1,  41,  46,  44,  45,  47,  48,  49,  50,  52,  53,
         54,  55,  56,  57,  58,  59,  -1,  62,  63,  -1,  -1,  -1,  71,
         68,  67,  70,  72,  73,  78,  -1,  74,  75,  76,  77,  83,  79,
         85,  81,  84,  87,  -1,  86,  -1,  90,  88,  91,  93,  99,  97,
         95, 101,  98,  96,  94, 102, 103, 106, 105, 107, 109, 110, 108,
        113,  -1, 111, 114, 112, 116, 117, 118, 125, 119, 123, 124,  -1,
        121, 120, 122, 126, 128, 127, 129, 130, 132, 136, 131, 133, 135,
         -1, 137, 134,  -1,  -1, 138, 139, 141, 140, 142, 143, 148, 149,
        151, 144, 156, 150, 154, 145, 152,  -1, 155, 146,  -1, 157, 159,
        158, 160,  -1, 163, 162, 164, 167, 169, 168, 166, 175,  -1, 174,
        170, 172, 177, 173, 176, 178, 179, 181, 182

dtype('int64')